# Parte 3 – Cruce por ubigeo y análisis

**Pregunta del trabajo:** ¿El Estado declara emergencia donde más llueve?

**Temporada asignada:** 1 de enero al 31 de mayo de 2023 (fechas de publicación de los decretos y de la lluvia).

In [3]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import plotly.express as px

DATOS = Path("datos")
DATOS.mkdir(exist_ok=True)

## 1. Crear `datos/ubigeos.csv`

Usamos los 25 ubigeos departamentales del INEI (24 departamentos + Callao).

 Si hubiéramos copiado esos nombres tal cual, Áncash y Junín no habrían encontrado su ubigeo en el paso 4. Por eso escribimos los nombres oficiales, con tildes, igual que en la lista de departamentos de la Parte 1.

El ubigeo se construye como **texto** con `f"{i:02d}"`, así el `1` se guarda como `"01"`.

In [29]:
nombres = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
           "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
           "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
           "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
           "San Martín", "Tacna", "Tumbes", "Ucayali"]

ubigeos = pd.DataFrame({
    "ubigeo": [f"{i:02d}" for i in range(1, 26)],   # "01", "02", ..., "25"
    "departamento": nombres,
})
ubigeos.to_csv(DATOS / "ubigeos.csv", index=False, encoding="utf-8-sig")

print("Filas:", len(ubigeos))


Filas: 25


## 2. Leer el archivo dos veces

Una vez con `pd.read_csv` normal y otra con `dtype={"ubigeo": str}`.

In [30]:
ubi_normal = pd.read_csv(DATOS / "ubigeos.csv")
ubi_texto = pd.read_csv(DATOS / "ubigeos.csv", dtype={"ubigeo": str})

print("Tipo de 'ubigeo' con lectura normal :", ubi_normal["ubigeo"].dtype)
print("Tipo de 'ubigeo' con dtype=str      :", ubi_texto["ubigeo"].dtype)

pd.DataFrame({
    "departamento": ubi_texto["departamento"],
    "lectura normal": ubi_normal["ubigeo"],
    "con dtype=str": ubi_texto["ubigeo"],
}).head(10)

Tipo de 'ubigeo' con lectura normal : int64
Tipo de 'ubigeo' con dtype=str      : str


,departamento,lectura normal,con dtype=str
0,Amazonas,1,01
1,Áncash,2,02
2,Apurímac,3,03
3,Arequipa,4,04
4,Ayacucho,5,05
5,Cajamarca,6,06
6,Callao,7,07
7,Cusco,8,08
8,Huancavelica,9,09
9,Huánuco,10,10


 **La diferencia:** 
 
 Leer la Dataframe con `pd.read_csv` normal, la lee como numero, el ubigeo que es 01  la convierte a entero 1,  es decir que el cero desaparece en los departamentos con ubigeo 01 al 09(de Amazonas a Huancavelica).  y otra con `dtype={"ubigeo": str}`, mantiene el cero

**¿Por qué importa?**

- El ubigeo **no es una cantidad, es un código** oficial. Los ubigeos de provincia y distrito se construyen pegando códigos (`01` + `01` + `01` = `010101`)
- Para no perder el cero. Al cruzar tablas, un `1` (número) y un `"01"` (texto) **no son iguales**: pandas directamente da error.
- Perderiamos datos, y aunque se convierta el número a texto con `.astype(str)`, solo coinciden 16 de 25 (los que ya tenían dos dígitos). Los 9 primeros departamentos se perderían sin aviso.
- Por eso, en todo el notebook leemos los ubigeos siempre con `dtype={"ubigeo": str}`.

## 3. Agregar el ubigeo a las tablas de decretos y de lluvias


In [6]:
decretos = (pd.read_csv(DATOS / "decretos_por_departamento.csv", encoding="utf-8-sig")
              .drop(columns="ubigeo", errors="ignore"))
lluvias = (pd.read_csv(DATOS / "lluvias_por_departamento.csv", encoding="utf-8-sig")
             .drop(columns="ubigeo", errors="ignore"))
decretos_lluvias = pd.read_csv(DATOS / "decretos_lluvias.csv", encoding="utf-8-sig")

print("decretos_por_departamento:", decretos.shape)
print("lluvias_por_departamento :", lluvias.shape)
print("decretos_lluvias         :", decretos_lluvias.shape)

intento_decretos = decretos.merge(ubi_texto, on="departamento", how="left")
intento_lluvias = lluvias.merge(ubi_texto, on="departamento", how="left")
intento_lluvias.head()

decretos_por_departamento: (25, 3)
lluvias_por_departamento : (25, 6)
decretos_lluvias         : (21, 6)


,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,ubigeo
0,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,01
1,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,02
2,Apurímac,Abancay,-13.63426,-72.88380,726.2,3,03
3,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1,04
4,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2,05


## 4. Verificación: ¿qué departamentos no encontraron su ubigeo?

In [7]:
def sin_ubigeo(tabla):
    return tabla.loc[tabla["ubigeo"].isna(), "departamento"].tolist()

print("Decretos sin ubigeo (nombre exacto):", sin_ubigeo(intento_decretos))
print("Lluvias  sin ubigeo (nombre exacto):", sin_ubigeo(intento_lluvias))

Decretos sin ubigeo (nombre exacto): []
Lluvias  sin ubigeo (nombre exacto): []


Con los archivos actuales el cruce por nombre exacto funciona, porque la Parte 1 usó la lista de departamentos con tildes del enunciado y la Parte 2 tomó los nombres de Wikipedia (que también llevan tildes) y agregó el Callao.

Pero es frágil: basta que alguien escriba *"Ancash"* sin tilde, *"JUNIN"* en mayúsculas, o que se copie el `admin1` de la API de Open-Meteo (*"Departamento de Cusco"*, *"Provincia Constitucional del Callao"*) para que el departamento se quede sin ubigeo. Además, una misma letra con tilde se puede guardar de dos formas en Unicode (`á` como un solo carácter, o `a` + tilde), y a la vista son idénticas pero para Python son distintas.

Para que el cruce aguante esos casos, en vez de comparar el nombre tal cual comparamos una **clave normalizada**: sin tildes, en minúsculas, sin espacios de más y sin prefijos como *"Departamento de"*. Si aparece un nombre que ni así se reconoce, se agrega a mano en el diccionario `correcciones`.

In [32]:
# Nombres alternativos que la normalización sola no resuelve
correcciones = {
    "cuzco": "cusco",
}

def normalizar(nombre):
    texto = unicodedata.normalize("NFKD", str(nombre))
    texto = "".join(c for c in texto if not unicodedata.combining(c))  # quita tildes
    texto = " ".join(texto.lower().split())                             # minúsculas y espacios
    for prefijo in ["departamento de ", "departamento del ",
                    "provincia constitucional del ", "region "]:
        if texto.startswith(prefijo):
            texto = texto[len(prefijo):]
    return correcciones.get(texto, texto)

# Probamos la función con variantes que podrían venir en los archivos
pruebas = ["Áncash", "Ancash", "ANCASH", "  Huánuco ", "Departamento de Junín",
           "Departamento del Cusco", "Cuzco", "Provincia Constitucional del Callao",
           "San Martin"]
pd.DataFrame({"escrito así": pruebas, "clave": [normalizar(p) for p in pruebas]})

,escrito así,clave
0,Áncash,ancash
1,Ancash,ancash
2,ANCASH,ancash
3,Huánuco,huanuco
4,Departamento de Junín,junin
5,Departamento del Cusco,cusco
6,Cuzco,cusco
7,Provincia Constitucional del Callao,callao
8,San Martin,san martin


In [9]:
ubi_texto["clave"] = ubi_texto["departamento"].apply(normalizar)
assert ubi_texto["clave"].is_unique   # la clave no debe juntar dos departamentos

def agregar_ubigeo(tabla):
    tabla = tabla.copy()
    tabla["clave"] = tabla["departamento"].apply(normalizar)
    tabla = tabla.merge(ubi_texto[["clave", "ubigeo"]], on="clave", how="left")
    return tabla.drop(columns="clave")

decretos = agregar_ubigeo(decretos)
lluvias = agregar_ubigeo(lluvias)

print("Decretos sin ubigeo (clave normalizada):", sin_ubigeo(decretos))
print("Lluvias  sin ubigeo (clave normalizada):", sin_ubigeo(lluvias))

# Verificaciones: nadie sin ubigeo y ningún ubigeo repetido
assert decretos["ubigeo"].notna().all() and lluvias["ubigeo"].notna().all()
assert decretos["ubigeo"].is_unique and lluvias["ubigeo"].is_unique
# La tabla de lluvias debe tener los 25 departamentos
assert set(lluvias["ubigeo"]) == set(ubi_texto["ubigeo"])
print("OK: todos los departamentos tienen ubigeo y ninguno se repite.")

Decretos sin ubigeo (clave normalizada): []
Lluvias  sin ubigeo (clave normalizada): []
OK: todos los departamentos tienen ubigeo y ninguno se repite.


Guardamos los dos archivos con la columna `ubigeo` agregada al inicio (como texto de 2 dígitos).

In [33]:
def ubigeo_primero(tabla):
    return tabla[["ubigeo"] + [c for c in tabla.columns if c != "ubigeo"]]

decretos = ubigeo_primero(decretos).sort_values("ubigeo").reset_index(drop=True)
lluvias = ubigeo_primero(lluvias).sort_values("ubigeo").reset_index(drop=True)

decretos.to_csv(DATOS / "decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
lluvias.to_csv(DATOS / "lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")
decretos.head()

,ubigeo,departamento,declaratorias,prorrogas
0,01,Amazonas,3,1
1,02,Áncash,6,1
2,03,Apurímac,1,0
3,04,Arequipa,4,0
4,05,Ayacucho,5,1


## 5. Cruzar las dos tablas por ubigeo (no por nombre)

Partimos de la tabla de **lluvias**, que tiene los 25 departamentos, y le pegamos los decretos con `how="left"`. Así ningún departamento se pierde aunque no tenga decretos.

- `validate="one_to_one"` hace que pandas dé error si un ubigeo estuviera repetido en alguna de las dos tablas.
- `indicator=True` agrega la columna `_merge`, que dice si cada fila encontró pareja (`both`) o solo estaba en lluvias (`left_only`).

In [37]:
tabla = lluvias.merge(
    decretos[["ubigeo", "declaratorias", "prorrogas"]],
    on="ubigeo", how="left", validate="one_to_one", indicator=True,
)
print(tabla["_merge"].value_counts(), "\n")
print("Solo en lluvias (sin decretos):",
      tabla.loc[tabla["_merge"] == "left_only", "departamento"].tolist())

# Para comparar: con un cruce "inner" esos departamentos desaparecerían
print("Filas si se usara how='inner':", len(lluvias.merge(decretos, on="ubigeo")))

_merge
both          25
left_only      0
right_only     0
Name: count, dtype: int64 

Solo en lluvias (sin decretos): []
Filas si se usara how='inner': 25


In [39]:
# Sin decretos = 0 declaratorias y 0 prórrogas (no NaN)
tabla[["declaratorias", "prorrogas"]] = tabla[["declaratorias", "prorrogas"]].fillna(0).astype(int)

# Nombre oficial del departamento, tomado de la tabla de ubigeos
tabla["departamento"] = tabla["ubigeo"].map(ubi_texto.set_index("ubigeo")["departamento"])

tabla = (tabla.drop(columns="_merge",errors="ignore")
              .sort_values("ubigeo")
              .reset_index(drop=True)
              [["ubigeo", "departamento", "capital", "latitud", "longitud",
                "lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias", "prorrogas"]])

## 6. Verificación: 25 filas y los departamentos sin declaratorias con 0

In [40]:
print("Filas de la tabla final:", len(tabla))
assert len(tabla) == 25
assert tabla["ubigeo"].is_unique
assert tabla[["declaratorias", "prorrogas"]].notna().all().all()

# El cruce no debe perder ni inventar declaratorias
assert tabla["declaratorias"].sum() == decretos["declaratorias"].sum()
assert tabla["prorrogas"].sum() == decretos["prorrogas"].sum()
print("Total de declaratorias (departamento × decreto):", tabla["declaratorias"].sum())
print("Total de prórrogas     (departamento × decreto):", tabla["prorrogas"].sum())

print("\nDepartamentos con 0 declaratorias:")
tabla[tabla["declaratorias"] == 0]

Filas de la tabla final: 25
Total de declaratorias (departamento × decreto): 73
Total de prórrogas     (departamento × decreto): 6

Departamentos con 0 declaratorias:


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
16,17,Madre de Dios,Puerto Maldonado,-12.58946,-69.19948,1248.4,23,0,0


In [41]:
tabla

,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,3,1
1,02,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,6,1
2,03,Apurímac,Abancay,-13.63426,-72.88380,726.2,3,1,0
3,04,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1,4,0
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2,5,1
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,796.6,0,4,0
6,07,Callao,Callao,-12.05162,-77.13452,55.5,0,1,0
7,08,Cusco,Cusco,-13.53188,-71.96701,714.0,1,2,0
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,817.8,3,3,0
9,10,Huánuco,Huánuco,-9.92882,-76.23989,524.3,2,4,0


## 7. Guardar `datos/tabla_final.csv`

El CSV guarda el texto `01` tal cual; lo importante es que al volver a leerlo se use `dtype={"ubigeo": str}`. Lo comprobamos leyendo el archivo de nuevo y mirando sus primeras líneas en crudo.

(Ojo: si se abre el CSV con doble clic en Excel, Excel muestra `1` porque vuelve a interpretar la columna como número. El archivo está bien; es Excel el que lo muestra así. Para verlo bien en Excel hay que importarlo marcando la columna como *Texto*.)

In [15]:
tabla.to_csv(DATOS / "tabla_final.csv", index=False, encoding="utf-8-sig")

comprobacion = pd.read_csv(DATOS / "tabla_final.csv", dtype={"ubigeo": str})
assert comprobacion.shape == (25, 9)
assert comprobacion["ubigeo"].str.fullmatch(r"\d{2}").all()   # siempre 2 dígitos
print("Primeros ubigeos leídos:", comprobacion["ubigeo"].head(3).tolist())

with open(DATOS / "tabla_final.csv", encoding="utf-8-sig") as archivo:
    for linea in archivo.read().splitlines()[:4]:
        print(linea)

Primeros ubigeos leídos: ['01', '02', '03']
ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
01,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,3,1
02,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,6,1
03,Apurímac,Abancay,-13.63426,-72.8838,726.2,3,1,0


## 8. Gráficos con Plotly

**Dispersión.** Cada punto es un departamento: en X la lluvia acumulada en su capital (1 de enero – 31 de mayo de 2023) y en Y el número de declaratorias de emergencia por lluvias que lo mencionan. Como muchos departamentos tienen el mismo número de declaratorias, algunos nombres quedarían encimados; cuando dos puntos de la misma altura están muy cerca, alternamos su etiqueta arriba/abajo. Al pasar el mouse se ven la capital, los días de lluvia fuerte y las prórrogas.

In [42]:
AZUL = "#2a78d6"
GRIS = "#52514e"
FUENTE = ("Fuente: decretos supremos de la PCM en gob.pe y lluvia diaria de Open-Meteo "
          "(1 ene – 31 may 2023). Elaboración propia.")

# Etiquetas: si dos puntos con las mismas declaratorias están cerca en X,
# la segunda etiqueta va abajo para que no se encimen.
graf = tabla.sort_values(["declaratorias", "lluvia_total_mm"]).reset_index(drop=True)
umbral = 0.08 * graf["lluvia_total_mm"].max()
posiciones = []
for i, fila in graf.iterrows():
    cerca = (i > 0
             and graf.loc[i - 1, "declaratorias"] == fila["declaratorias"]
             and fila["lluvia_total_mm"] - graf.loc[i - 1, "lluvia_total_mm"] < umbral)
    if cerca and posiciones[-1] == "top center":
        posiciones.append("bottom center")
    else:
        posiciones.append("top center")

fig_dispersion = px.scatter(
    graf, x="lluvia_total_mm", y="declaratorias", text="departamento",
    hover_name="departamento",
    hover_data={"departamento": False, "capital": True,
                "lluvia_total_mm": ":.1f", "dias_lluvia_fuerte": True,
                "declaratorias": True, "prorrogas": True},
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)",
            "declaratorias": "Declaratorias de emergencia por lluvias",
            "dias_lluvia_fuerte": "Días con 20 mm o más",
            "prorrogas": "Prórrogas", "capital": "Capital"},
    title="¿Más lluvia, más declaratorias? Lluvia vs. declaratorias por departamento, ene–may 2023",
)
fig_dispersion.update_traces(
    marker=dict(size=11, color=AZUL, line=dict(width=2, color="white")),
    textposition=posiciones, textfont=dict(size=11, color=GRIS),
    cliponaxis=False,
)
fig_dispersion.update_layout(
    template="plotly_white", height=600, title_font_size=17,
    margin=dict(l=10, r=30, t=70, b=90),
    xaxis=dict(rangemode="tozero"),
    yaxis=dict(range=[-0.6, graf["declaratorias"].max() + 0.8], dtick=1),
    annotations=[dict(text=FUENTE, showarrow=False, xref="paper", yref="paper",
                      x=0, y=-0.16, xanchor="left", font=dict(size=11, color="gray"))],
)
fig_dispersion.show()

**Barras.** Los 10 departamentos con más declaratorias. Hay **empate en el último puesto**: varios departamentos tienen el mismo número de declaratorias que el décimo. Para desempatar usamos primero las **prórrogas** (si la emergencia se tuvo que prorrogar, el problema duró más) y luego el orden alfabético. Los departamentos empatados que quedan fuera se indican debajo del gráfico, para no esconder el empate.

In [44]:
orden = tabla.sort_values(["declaratorias", "prorrogas", "departamento"],
                          ascending=[False, False, True])
top10 = orden.head(10)
corte = top10["declaratorias"].iloc[-1]
empatados_fuera = orden.iloc[10:].query("declaratorias == @corte")["departamento"].tolist()
print(f"El décimo puesto tiene {corte} declaratorias.")
print("Empatados con el décimo que quedaron fuera:", empatados_fuera)

nota = FUENTE
if empatados_fuera:
    nota = (f"También tienen {corte} declaratorias: {', '.join(empatados_fuera)} "
            f"(desempate por prórrogas y luego orden alfabético).<br>" + FUENTE)

fig_barras = px.bar(
    top10.iloc[::-1], x="declaratorias", y="departamento", orientation="h",
    text="declaratorias",
    hover_data={"prorrogas": True, "lluvia_total_mm": ":.1f"},
    labels={"declaratorias": "Declaratorias de emergencia por lluvias",
            "departamento": "", "prorrogas": "Prórrogas",
            "lluvia_total_mm": "Lluvia en la capital (mm)"},
    title="Los 10 departamentos con más declaratorias de emergencia por lluvias, ene–may 2023",
)
fig_barras.update_traces(marker_color=AZUL, textposition="outside",
                         textfont=dict(color=GRIS), cliponaxis=False)
fig_barras.update_layout(
    template="plotly_white", height=520, title_font_size=17, barcornerradius=4,
    bargap=0.25, margin=dict(l=10, r=30, t=70, b=130),
    xaxis=dict(dtick=1, range=[0, top10["declaratorias"].max() + 0.7]),
    annotations=[dict(text=nota, showarrow=False, xref="paper", yref="paper",
                      x=0, y=-0.17, xanchor="left", yanchor="top", align="left",
                      font=dict(size=11, color="gray"))],
)
fig_barras.show()

El décimo puesto tiene 3 declaratorias.
Empatados con el décimo que quedaron fuera: ['Piura', 'Huancavelica', 'Ica', 'Ucayali']


## 9. Análisis

Primero calculamos los números que usamos en las respuestas.

In [50]:
# a) Top 3 por declaratorias (keep="all" para no esconder empates) y top 3 por lluvia
top3_declaratorias = tabla.nlargest(3, "declaratorias", keep="all")
top3_lluvia = tabla.nlargest(4, "lluvia_total_mm", keep="all")

print("Top 3 por declaratorias:")
print(top3_declaratorias[["departamento", "declaratorias", "lluvia_total_mm"]].to_string(index=False))
print("\nTop 4 por lluvia acumulada:")
print(top3_lluvia[["departamento", "lluvia_total_mm", "declaratorias"]].to_string(index=False))
print("\nCoinciden:", set(top3_declaratorias["departamento"]) & set(top3_lluvia["departamento"]) or "ninguno")

Top 3 por declaratorias:
departamento  declaratorias  lluvia_total_mm
        Lima              7             27.9
      Áncash              6           1145.8
    Ayacucho              5            498.4
 La Libertad              5            177.2

Top 4 por lluvia acumulada:
 departamento  lluvia_total_mm  declaratorias
       Loreto           1604.6              1
      Ucayali           1316.0              3
Madre de Dios           1248.4              0
       Áncash           1145.8              6

Coinciden: {'Áncash'}


In [21]:
# ¿Hay relación entre lluvia y declaratorias? Correlación de Spearman
# (usa rankings: sirve con pocos datos y con valores enteros repetidos)
tabla[["lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias"]].corr(method="spearman").round(2)

,lluvia_total_mm,dias_lluvia_fuerte,declaratorias
lluvia_total_mm,1.00,0.82,-0.19
dias_lluvia_fuerte,0.82,1.00,-0.20
declaratorias,-0.19,-0.20,1.00


In [51]:
# b) Mucha lluvia y pocas declaratorias, o al revés.
# Cortamos por la mediana de lluvia y por la mediana de declaratorias.
med_lluvia = tabla["lluvia_total_mm"].median()
med_decl = tabla["declaratorias"].median()
print(f"Mediana de lluvia: {med_lluvia:.1f} mm | mediana de declaratorias: {med_decl:.0f}")

columnas = ["departamento", "capital", "lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias"]

mucha_lluvia_pocas = tabla[(tabla["lluvia_total_mm"] > med_lluvia) & (tabla["declaratorias"] < med_decl)]
print("\nMucha lluvia (sobre la mediana) y declaratorias bajo la mediana:")
print(mucha_lluvia_pocas[columnas].sort_values("lluvia_total_mm", ascending=False).to_string(index=False))

poca_lluvia_muchas = tabla[(tabla["lluvia_total_mm"] < med_lluvia) & (tabla["declaratorias"] > med_decl)]
print("\nPoca lluvia (bajo la mediana) y declaratorias sobre la mediana:")
print(poca_lluvia_muchas[columnas].sort_values("declaratorias", ascending=False).to_string(index=False))

Mediana de lluvia: 714.0 mm | mediana de declaratorias: 3

Mucha lluvia (sobre la mediana) y declaratorias bajo la mediana:
 departamento          capital  lluvia_total_mm  dias_lluvia_fuerte  declaratorias
       Loreto          Iquitos           1604.6                  26              1
Madre de Dios Puerto Maldonado           1248.4                  23              0
       Tumbes           Tumbes            931.4                  14              2
        Junín         Huancayo            850.8                   6              2
        Pasco   Cerro de Pasco            750.4                   1              2
     Apurímac          Abancay            726.2                   3              1

Poca lluvia (bajo la mediana) y declaratorias sobre la mediana:
departamento  capital  lluvia_total_mm  dias_lluvia_fuerte  declaratorias
        Lima   Huacho             27.9                   0              7
 La Libertad Trujillo            177.2                   1              5
    Ayac

In [52]:
# c) Peligro inminente vs. impacto de daños (tabla de decretos de la Parte 1)
declaratorias = decretos_lluvias[decretos_lluvias["tipo"] == "declara"].copy()
print("Decretos de lluvias:", len(decretos_lluvias), "| declaratorias:", len(declaratorias),
      "| prórrogas:", (decretos_lluvias["tipo"] == "prorroga").sum())

motivos = pd.DataFrame({
    "decretos": declaratorias["motivo"].value_counts(),
    "porcentaje": declaratorias["motivo"].value_counts(normalize=True).mul(100).round(1),
})
motivos

Decretos de lluvias: 21 | declaratorias: 18 | prórrogas: 3


,decretos,porcentaje
motivo,,
impacto de daños,14,77.8
peligro inminente,3,16.7
otro,1,5.6


In [45]:
# ¿Qué decretos quedaron como "otro"?
for _, fila in declaratorias[declaratorias["motivo"] == "otro"].iterrows():
    print(fila["numero"], "|", fila["fecha"])
    print(fila["titulo"], "\n")

Decreto Supremo N.° 043-2023-PCM | 2023-03-26
Decreto Supremo que declara el Estado de Emergencia Nacional, por desastre de gran magnitud, a consecuencia de intensas precipitaciones pluviales en los departamentos de Lambayeque, Piura y Tumbes 



In [54]:
# Un decreto puede cubrir muchos departamentos. ¿Cuántos cubre cada tipo de decreto?
# (misma regla de la Parte 1: título original y palabra completa; set() cuenta
#  una sola vez "provincia de Ica del departamento de Ica")
def departamentos_de(titulo):
    return {d for d in ubi_texto["departamento"] if re.search(r"\b" + d + r"\b", titulo)}

declaratorias["n_departamentos"] = declaratorias["titulo"].apply(lambda t: len(departamentos_de(t)))

# Control cruzado con la Parte 1: la suma debe dar el total de la tabla final
assert declaratorias["n_departamentos"].sum() == tabla["declaratorias"].sum()

cobertura = declaratorias.groupby("motivo")["n_departamentos"].agg(
    decretos="count", departamentos_cubiertos="sum", promedio_por_decreto="mean")
cobertura["% de departamentos cubiertos"] = (
    cobertura["departamentos_cubiertos"] / cobertura["departamentos_cubiertos"].sum() * 100).round(1)
cobertura.round(1)

,decretos,departamentos_cubiertos,promedio_por_decreto,% de departamentos cubiertos
motivo,,,,
impacto de daños,14,33,2.4,45.2
otro,1,3,3.0,4.1
peligro inminente,3,37,12.3,50.7


## Respuestas

*Todos los números que siguen salen de las celdas de código de arriba.*

### a) ¿Cuáles fueron los 3 departamentos con más declaratorias? ¿Coinciden con los 3 donde más llovió?

Los departamentos con más declaratorias por lluvias fueron **Lima(7), Áncash(6)** y el tercer lugar es un **empate entre Ayacucho y La Libertad, con 5 declaratorias**

Los 3 con más lluvia fueron **Loreto (1604,6 mm en Iquitos), Ucayali (1316,0 mm en Pucallpa) y Madre de Dios (1248,4 mm en Puerto Maldonado)**.

**No coincide ninguno.** Los tres departamentos más lluviosos tuvieron 1, 3 y 0 declaratorias. Al revés, Lima empata en el primer lugar de declaratorias y con menos lluvia de la temporada (55,5 mm). El único caso donde coinciden mucha lluvia y muchas declaratorias es Áncash (cuarto en lluvia y segundo en declaratorias). 

### b) ¿Hay departamentos con mucha lluvia y pocas o ninguna declaratoria, o al revés?

Sí, en los dos sentidos. Separando por la mediana (714 mm de lluvia y 3 declaratorias):

- **Mucha lluvia y pocas declaratorias:** Loreto (1604,6 mm y 26 días con 20 mm o más, pero 1 declaratoria), Madre de Dios (1248,4 mm y 23 días fuertes, **ninguna** declaratoria), Tumbes (931,4 mm, 2), y también Junín, Pasco y Apurímac.
- **Poca lluvia y muchas declaratorias:** Lima (55,5 mm, ningún día fuerte, 7 declaratorias), La Libertad (177,2 mm en Trujillo, 5), Arequipa (362,0 mm, 4), Ayacucho (498,4 mm, 5) y Huánuco (524,3 mm, 4).

**Razones posibles:**

1. **Lo que causa daño es la lluvia fuera de lo normal, no la cantidad total.** En Iquitos, Pucallpa o Puerto Maldonado es normal que llueva más de 1000 mm entre enero y mayo: las viviendas, los cultivos y las ciudades están adaptados a eso. En la costa, donde casi no llueve, 50 a 200 mm bastan para causar daños porque infraestructura no adecuadas, no hay drenajes ni techos preparados. En 2023, además, la costa norte y centro tuvo lluvias muy inusuales en marzo.

3. **La vulnerabilidad y la exposición cuentan más que la lluvia.** Donde hay más población, viviendas en quebradas o cauces de ríos, y carreteras, la misma lluvia causa más daños y más pedidos de emergencia.



### c) ¿Qué porcentaje de las declaratorias fue por peligro inminente y qué porcentaje por impacto de daños?

De las **18 declaratorias** por lluvias de la temporada (sin contar las 3 prórrogas):

| Motivo | Decretos | % |
|---|---|---|
| Impacto de daños (después) | 14 | **77,8 %** |
| Peligro inminente (antes) | 3 | **16,7 %** |
| Otro | 1 | 5,6 % |


De las 18 declaratorias(sin contar las 3 porrogas), el 77,8% (14) fueron por impacto de daños, es decir reactivas despues de huaico/desbordes.Solo el 16,7% fueron por peligro inminente, preventivas. si sumamos el  el DS 043-2023 que es por desastre de "gran magnitud" tambien reactivo, el 83,4% fueron reactivads

El caso "otro" es el DS 043-2023-PCM (emergencia nacional *por desastre de gran magnitud*). Su título no usa ninguna de las dos frases, pero también se dio después del daño; si lo sumamos a impacto de daños, **el 82 % de las declaratorias fueron reactivas**.


### d) Limitaciones del análisis

1. **La lluvia de todo un departamento se representa con un solo punto: su capital.** Los decretos, en cambio, se refieren a distritos concretos, que muchas veces están lejos de la capital y a otra altura (el caso de Yauyos en Lima).
2. **Usamos la lluvia total y no la anomalía.** No comparamos con lo que llueve normalmente en cada lugar, que es lo que explica mejor si una lluvia es peligrosa.
3. **La clasificación depende del título del decreto.** Un decreto por lluvias cuyo título no diga "lluvia" ni "precipitaciones" queda fuera; por ejemplo, el título completo del DS 036-2023-PCM es solo "Declaratoria de Estado de Emergencia", y sin leer el PDF no sabemos por qué fue. Además, dependemos de lo que devuelve el buscador de gob.pe para el término de búsqueda: si un decreto está indexado con otras palabras, no aparece.


### e) Conclusión: ¿el Estado declara emergencia donde más llueve?

No. Entre enero y mayo de 2023 las declaratorias de emergencia por lluvias no siguieron a la cantidad de lluvia: los tres departamentos más lluviosos (Loreto, Ucayali y Madre de Dios) sumaron solo 4 declaratorias, mientras que Lima, lidera con 7, con una de las lluvias más bajas del país, seguida con Áncash (6), y la correlación entre ambas variables es prácticamente nula (−0,16). Las declaratorias parecen seguir al **daño** y no a los milímetros: aparecen donde la lluvia es inusual para la zona (la costa y los valles andinos), donde hay más población e infraestructura expuesta, y donde los gobiernos regionales las tramitan. Además, 3 de cada 4 declaratorias se dieron por impacto de daños, es decir, cuando el desastre ya había ocurrido. Con las limitaciones señaladas (lluvia medida solo en la capital y con datos de modelo), nuestra respuesta es que el Estado declara emergencia donde la lluvia hace daño, no donde más llueve.